# Tutorial: Plan 004.2 Current Observation API

**Audience:** Python beginners who know variables, functions, and lists.

**Prerequisites:** the `archiver` development environment and basic use of `Path`.

**Learning goals:** build immutable current-file queries, inspect expression objects, and fetch bounded results from the latest successful scan.

**Outline:** setup → expressions → immutable queries → fetch → exercise → pitfalls.

This notebook uses only the supported public API. Its temporary files and catalog are deleted by the final cell.


In [1]:
from __future__ import annotations

import os
from pathlib import Path
from tempfile import TemporaryDirectory

from archiver import Catalog, CurrentFileQuery
from archiver.query import mtime, path, size


temporary_directory = TemporaryDirectory(prefix="archiver-current-query-")
workspace = Path(temporary_directory.name)
root = workspace / "source"
root.mkdir()

examples = (
    (root / "a-note.txt", b"a", 1_700_000_000_000_000_000),
    (root / "b-report.txt", b"bb", 1_700_000_001_000_000_000),
    (root / "nested" / "c-report.txt", b"ccc", 1_700_000_002_000_000_000),
    (root / "nested" / "data.bin", b"dddd", 1_700_000_003_000_000_000),
)
for file_path, contents, observed_mtime in examples:
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_bytes(contents)
    os.utime(file_path, ns=(observed_mtime, observed_mtime))

catalog = Catalog.create(workspace / "catalog.sqlite")
catalog.scan_directory(root)


ScanSummary(files_observed=4, total_bytes_observed=10, distinct_content_count=4, duplicate_content_group_count=0)

## 1. Fields create expression objects

`path`, `size`, and `mtime` are field objects. Calling their methods creates frozen value objects that **describe intent**; it does not access SQLite or scan the filesystem.


In [54]:
text_files = path.glob("*.txt")
largest_first = size.desc()
newest_first = mtime.desc()

expressions = (text_files, largest_first, newest_first)

assert text_files == path.glob("*.txt")
assert "relative_path" not in repr(expressions)
assert "size_bytes" not in repr(expressions)
assert "mtime_ns" not in repr(expressions)

for exp in expressions:
    print(exp)
#text_files.field.name
#for attr in dir(text_files):
#    if not attr.startswith("_"):
#        print(f"{attr}: {getattr(text_files, attr)}")

PathGlobPredicate(field=<_CurrentFileField.PATH: 'path'>, pattern='*.txt')
SortExpression(field=<_CurrentFileField.SIZE: 'size'>, direction=<_SortDirection.DESCENDING: 'descending'>)
SortExpression(field=<_CurrentFileField.MTIME: 'mtime'>, direction=<_SortDirection.DESCENDING: 'descending'>)


## 2. Query methods return new queries

`Catalog.files(root)` creates a lazy query with path-ascending order and a limit of 20. `where()`, `sort()`, and `limit()` use immutable replacement: each call returns a new query and leaves the earlier object reusable.


In [34]:
base = catalog.files(root)
nested_text = base.where(path.glob("nested/*.txt"))
largest_nested_text = nested_text.sort(size.desc()).limit(1)

print(base,'\n')
print(nested_text, '\n')
print(largest_nested_text)

assert isinstance(base, CurrentFileQuery)
assert base.predicates == ()
assert base.row_limit == 20
assert nested_text.row_limit == 20
assert largest_nested_text.row_limit == 1


CurrentFileQuery(root=WindowsPath('C:/Users/pmbarakm/AppData/Local/Temp/archiver-current-query-wvac779g/source'), predicates=(), ordering=SortExpression(field=<_CurrentFileField.PATH: 'path'>, direction=<_SortDirection.ASCENDING: 'ascending'>), row_limit=20) 

CurrentFileQuery(root=WindowsPath('C:/Users/pmbarakm/AppData/Local/Temp/archiver-current-query-wvac779g/source'), predicates=(PathGlobPredicate(field=<_CurrentFileField.PATH: 'path'>, pattern='nested/*.txt'),), ordering=SortExpression(field=<_CurrentFileField.PATH: 'path'>, direction=<_SortDirection.ASCENDING: 'ascending'>), row_limit=20) 

CurrentFileQuery(root=WindowsPath('C:/Users/pmbarakm/AppData/Local/Temp/archiver-current-query-wvac779g/source'), predicates=(PathGlobPredicate(field=<_CurrentFileField.PATH: 'path'>, pattern='nested/*.txt'),), ordering=SortExpression(field=<_CurrentFileField.SIZE: 'size'>, direction=<_SortDirection.DESCENDING: 'descending'>), row_limit=1)


## 3. `fetch()` interprets the description

The query stays lazy until `fetch()`. Results are a tuple of `FileObservation` objects from the latest successful scan. The default query is bounded even when the catalog contains more than 20 matches.


In [55]:
default_rows = base.fetch() #base.fetch()
default_summary = [
    (row.relative_path.as_posix(), row.size_bytes, row.mtime_ns)
    for row in default_rows
]

[
    print((row.relative_path.as_posix(), row.size_bytes, row.mtime_ns),'\n')
    for row in default_rows
]



assert isinstance(default_rows, tuple)
assert [item[0] for item in default_summary] == [
    "a-note.txt",
    "b-report.txt",
    "nested/c-report.txt",
    "nested/data.bin",
]


('a-note.txt', 1, 1700000000000000000) 

('b-report.txt', 2, 1700000001000000000) 

('nested/c-report.txt', 3, 1700000002000000000) 

('nested/data.bin', 4, 1700000003000000000) 



Get both nested files with a different pattern

In [58]:
all_nested_rows = base.where(path.glob("nested/*")).fetch()
[print(row.relative_path.as_posix()) for row in all_nested_rows];


nested/c-report.txt
nested/data.bin


## 4. Filtering, sorting, and limiting compose

Predicates are combined with AND. Sorting and limiting happen in SQLite before observations are materialized. Size and mtime ties use relative path as a deterministic tie-breaker.


In [50]:
query = (
    catalog.files(root)
    .where(path.glob("nested/*"))
    .where(path.glob("*.txt"))
    .sort(mtime.desc())
    .limit(2)
)
rows = query.fetch()
[row.relative_path.as_posix() for row in rows]

assert [row.relative_path.as_posix() for row in rows] == ["nested/c-report.txt"]
for row in rows:
    print(row.relative_path.as_posix())

print('\n \t Predicates:')
for p in  query.predicates:
    print(p,'')

nested/c-report.txt

 	 Predicates:
PathGlobPredicate(field=<_CurrentFileField.PATH: 'path'>, pattern='nested/*') 
PathGlobPredicate(field=<_CurrentFileField.PATH: 'path'>, pattern='*.txt') 


## 5. Exercise

Build a query for the two smallest `.txt` files anywhere below the root. Keep this base query unchanged:

```python
exercise_base = catalog.files(root)
```

Which field creates the predicate? Which field and direction describe "smallest first"? The next cell contains one solution.


In [43]:
exercise_base = catalog.files(root)
solution = exercise_base.where(path.glob("*.txt")).sort(size.asc()).limit(2)
solution_paths = tuple(row.relative_path.as_posix() for row in solution.fetch())
solution_paths

assert solution_paths == ("a-note.txt", "b-report.txt")
assert exercise_base.predicates == ()
assert exercise_base.row_limit == 20

solution.predicates

(PathGlobPredicate(field=<_CurrentFileField.PATH: 'path'>, pattern='*.txt'),)

## Pitfalls and mapping to Archiver

- `where()`, `sort()`, and `limit()` do not mutate the original query; keep the returned value.
- Limits must be positive integers. There is intentionally no public unbounded `fetch()` in Plan 004.2.
- GLOB patterns match the complete POSIX relative path and use SQLite GLOB rules; they are not Python regular expressions.
- `mtime` is the observed filesystem modification time, not scan time or content identity.
- `CurrentFileQuery` and its expression objects describe intent. `fetch()` asks the catalog to compile that typed description into parameterized SQL and materialize `FileObservation` values.

**Optional extension:** add two equal-sized files and verify that `size.asc()` orders the tie by relative path.


In [ ]:
catalog.close()
temporary_directory.cleanup()
print("Temporary tutorial data removed.")
